# Prepare real data

The experiment needs one row per patient/sample. Do not split cells from the same patient across train and test.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from stablewide.notebook_utils import find_project_root, run_experiment, read_outputs, core_subsample_table
ROOT = find_project_root(ROOT)
print(ROOT)

from stablewide.omics import validate_matrix_and_labels, pseudobulk_from_h5ad

## Option A: already have patient-by-gene CSVs

Place them under `data/processed/` and validate them here.

In [ ]:
X_CSV = ROOT / "data" / "processed" / "brca_mrna.csv"
Y_CSV = ROOT / "data" / "processed" / "brca_labels.csv"
LABEL_COL = "subtype"

if X_CSV.exists() and Y_CSV.exists():
    X, y, label_col = validate_matrix_and_labels(X_CSV, Y_CSV, LABEL_COL)
    print("X:", X.shape, "labels:", y[label_col].value_counts().to_dict())
else:
    print("CSV pair not present yet.")

## Option B: single-cell `.h5ad` -> patient pseudobulk

Set the three dataset-specific fields below. Counts are summed by patient and converted to log1p(CPM).

In [ ]:
H5AD = ROOT / "data" / "raw" / "breast_cancer.h5ad"
PATIENT_COL = "patient_id"   # change to the real obs column
LABEL_COL = "subtype"        # change if needed
COUNT_LAYER = "counts"       # or None to use adata.X

# Uncomment only after checking the metadata columns!!!
# X, y = pseudobulk_from_h5ad(
#     H5AD, X_CSV, Y_CSV,
#     patient_col=PATIENT_COL,
#     label_col=LABEL_COL,
#     layer=COUNT_LAYER,
#     min_cells=20,
# )
# print(X.shape, y[LABEL_COL].value_counts())

Keep patient IDs intact. With very small cohorts, the correct outcome may be “not enough evidence for a stable biomarker ranking.”